In [26]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

In [28]:
from SDRUtils.data.builder import SDRDataBuilder
from SDRUtils.products.usd import classify_sofr_swap_trade
from SDRUtils.products.usd.filters import new_sofr_swap_trades
from SDRUtils.packages import (
    detect_fly_trades_df,
    detect_curve_trades_df,
    detect_mms_trades_df,
    merge_package_legs_to_one_row,
)
from SDRUtils.core.classification import classifications_to_dataframe

In [29]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

start = NY_tz.localize(datetime.datetime(2025, 12, 29, 00, 1))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 20, 00))

raw_df = sdr.grab_sdr_trades(
    start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES", filter_func=new_sofr_swap_trades,
    # start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES", filter_func=filter_new_sofr_swaption_trades,
    # start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES",
)

swaps_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
curve = swaps_mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

CONCAT...: 100%|██████████| 2/2 [00:00<00:00, 247.01it/s]


In [30]:
from tqdm import tqdm

classifications = []
it = raw_df.iterrows()

it = tqdm(it, total=len(raw_df), desc="Classifying trades", unit="trade")

for idx, row in it:
	trade_id = int(row.get("Dissemination Identifier", idx))
	try:
		classification = classify_sofr_swap_trade(row, trade_id, curve)
		classifications.append(classification)
	except Exception as e:
		continue

classifications_df = classifications_to_dataframe(classifications)
with_pkg_df = merge_package_legs_to_one_row(detect_mms_trades_df(detect_curve_trades_df(detect_fly_trades_df(classifications_df))))

Classifying trades: 100%|██████████| 2121/2121 [00:11<00:00, 186.24trade/s]


In [34]:
with_pkg_df.tail(1).to_dict(orient="records")

[{'trade_id': 1574395035000000101,
  'execution_timestamp': Timestamp('2025-12-30 00:32:12+0000', tz='UTC'),
  'effective_date': Timestamp('2026-04-03 00:00:00'),
  'expiration_date': Timestamp('2030-05-31 00:00:00'),
  'product_type': 'OIS_SWAP',
  'tenor_years': 4.219444444444444,
  'tenor_label': '4Y',
  'is_forward': True,
  'forward_start_years': 0.2611111111111111,
  'forward_label': '3M',
  'trade_label': '3M 4Y',
  'notional': 82000000.0,
  'notional_currency': 'USD',
  'fixed_rate': 0.033477,
  'strike': nan,
  'estimated_pv01': 31438.779832216656,
  'package_type': 'SPREADOVER',
  'package_id': 'SPREADOVER_1574395035000000101',
  'package_legs': [1574395035000000101],
  'matched_ust_maturity': True,
  'ust_cusip': '91282CNG2',
  'ust_oi': '5-Year',
  'ust_issue_date': datetime.date(2025, 6, 2),
  'swap_maturity_date': datetime.date(2030, 5, 31)}]

In [37]:
# with_pkg_df["estimated_pv01"].sort_values(key=lambda x: float(str(x).split("/")[0]) if type(x) == str else float(x))

# with_pkg_df["pv01_clean"] = with_pkg_df["estimated_pv01"].apply(lambda x: float(str(x).split("/")[0]) if type(x) == str else float(x))
# with_pkg_df.sort_values(by="pv01_clean", ascending=False).head(10)